In [ ]:
using LensFactory
using LensFactory.Constants
using CairoMakie

## Initialize a cosmology

In [ ]:
# Initialize default cosmology
cosmo = Cosmology.init_cosmology()

# Lens and source redshifts
zl = 0.5
zs = 1.5

# ADDs and distance ratio
Dol = Cosmology.angular_diameter_distance(cosmo, 0., zl)
Dls = Cosmology.angular_diameter_distance(cosmo, zl, zs)
Dos = Cosmology.angular_diameter_distance(cosmo, 0., zs)
adis = Dls/Dos

## Create a grid
Since in most of the astrophysical scenario (primarily strong lensing by galaxies or galaxy clusters), the Einstein angle is ~[0.1, 50] arcseconds, it was decided to have `ANGLE_ARCSECOND` as default grid unit.

In [ ]:
# Create image plane grid (default units are ANGLE_ARCSEC)
x, y = Lenses.get_meshgrid(5, 5, 0.01);

## Point mass lens

In [ ]:
# Initialize an isolated point mass lens
lens = Lenses.init_PointLens(D_d = Dol, mass=1E11*MASS_SUN)

# Plot the image plane
fig, ax = Lenses.plot_image_plane(lens, x, y, adis)
display(fig)

## Composite lens: two point mass lenses

In [ ]:
# Initialize a two component lens
lens = Lenses.init_CompositeLens([(lens=:PointLens, D_d=Dol, x_c=0.0, y_c=0.0, mass=1E11*MASS_SUN),
                                  (lens=:PointLens, D_d=Dol, x_c=1.0, y_c=1.0, mass=1E11*MASS_SUN)])

# Plot the image plane (by default both source and image plane are shown in the same panel)
fig, ax = Lenses.plot_image_plane(lens, x, y, adis)
display(fig)

## Composite lens: ten randomly distributed point mass lenses
**Note:** By default, `LensFactory` determines the image position by finding the crossing points of lens equation. Hence, for singular lenses (such as `PointLens`), it can lead to spurious images at the lens position.

In [ ]:
using Random
Random.seed!(1234)

# Initialize a lens made of multiple point mass lenses
n_point = 10
ensamble = [(lens=:PointLens, D_d=Dol, x_c=(-3.0 + 6.0*rand()), y_c = (-3.0 + 6.0*rand()), mass=1E11*MASS_SUN) for _ in 1:n_point]
lens = Lenses.init_CompositeLens(ensamble)

# Let us have a point source
src = (0, 0.5)

# Here we have split the source and image plane for better visualization using "two_panel" keyword
fig, ax = Lenses.plot_image_plane(lens, x, y, adis; two_panel=true, source=src)
display(fig)


In [ ]:
# Let us add an extended source
src = Sources.gaussian(x, y, 0.1, 0.1, (0.0, 1.0), A=1)

# Plot the source and image plane
fig, ax = Lenses.plot_image_plane(lens, x, y, adis; two_panel=true, source=src)
display(fig)

In [ ]:
# Plot the magnification map in the image and source plane
fig, ax = Lenses.plot_magnification_map(lens, x, y, adis; plane=:image, heatmap_kws=(colorrange=(1,20),))
display(fig)

fig, ax = Lenses.plot_magnification_map(lens, x, y, adis; plane=:source, rays_per_pixel=100, heatmap_kws=(colorrange=(1,20),))
display(fig)

## Composite Lens: ten randomly distributed SIS lenses

In [ ]:
Random.seed!(1234)

# Initialize a lens made of multiple SIS lenses
n_point = 10
ensamble = [(lens=:SISLens, x_c=(-3.0+6.0*rand()), y_c = (-3.0+6.0*rand()), v_d=120E3) for _ in 1:n_point]
lens = Lenses.init_CompositeLens(ensamble)

# Point source position
src = (0.5, 1.0)

# Here we have split the source and image plane for better visualization using "two_panel" keyword
fig, ax = Lenses.plot_image_plane(lens, x, y, adis; two_panel=true, source=src)
display(fig)

In [ ]:
# Plot surface density
fig, ax = Lenses.plot_surface_density(lens, x, y, adis; D_d=Dol, unit=:convergence, 
                                             heatmap_kws=(colorrange=(0.1, 3.0), colormap=:cubehelix), 
                                             plot_contour=true,
                                             contour_kws=(linewidth=1, levels=0.5:0.2:1.5, color=:white))
display(fig)

## Composite lens: ten randomly distributed Plummer lenses

In [ ]:
# Initialize a lens made of multiple SIS lenses
Random.seed!(1234)

n_point = 10
ensamble = [(lens=:PlummerLens, D_d=Dol, x_c=(-3.0+6.0*rand()), y_c = (-3.0+6.0*rand()), x_s=0.3, mass=1E11*MASS_SUN) for _ in 1:n_point]
lens = Lenses.init_CompositeLens(ensamble)

# Here we have split the source and image plane for better visualization using "two_panel" keyword
fig, ax = Lenses.plot_image_plane(lens, x, y, adis; two_panel=true, source=(0.2, 1.0))
display(fig)

In [ ]:
fig, ax = Lenses.plot_surface_density(lens, x, y, adis; D_d=Dol, unit=:convergence, 
                                             heatmap_kws=(colorrange=(0.1, 3.0), colormap=:cubehelix), 
                                             plot_contour=true,
                                             contour_kws=(linewidth=1, levels=0.5:0.2:1.5, color=:white))
display(fig)

In [ ]:
fig, ax = Lenses.plot_magnification_map(lens, x, y, adis; plane=:image, heatmap_kws=(colorrange=(1,50), colormap=:viridis, colorscale=log10))
display(fig)

fig, ax = Lenses.plot_magnification_map(lens, x, y, adis; plane=:source, rays_per_pixel=10, heatmap_kws=(colorrange=(1,50), colormap=:viridis, colorscale=log10))
display(fig)

## 1D magnification profile (magnification vs area)

In [ ]:
fig, ax = Lenses.plot_magnification_profile(lens, x, y, adis; plane=:source, rays_per_pixel=10)
display(fig)